# Chapter 5: XGBoost (eXtreme Gradient Boosting)

> **Chapter Goal:** Chapter 4 showed GBM — powerful but slow, no native missing value handling, and easy to overfit without careful tuning. XGBoost is the same GBM math rewritten from scratch with three simultaneous upgrades: **smarter mathematics** (second-order derivatives), **built-in regularization** (in the objective itself), and **engineered for speed** (parallel column blocks, cache-aware access, histogram splits). Understanding WHY each upgrade was needed — and HOW it works — is the goal of this chapter.

---

## 1. Context: Why XGBoost Was Created

By 2014, Scikit-Learn's `GradientBoostingClassifier` had fundamental bottlenecks:

| Problem | Root Cause | Impact |
| :--- | :--- | :--- |
| **Slow on large data** | Single-threaded exact split search — sort $O(n \log n)$ per feature per node | Hours per training run on 1M+ rows |
| **No missing value support** | Must impute NaN before training | Extra preprocessing step, potential info loss |
| **Overfitting risk** | No regularization built into tree growth | Must tune depth/leaves carefully |
| **GBM only 1st order** | Uses gradient (slope) only, not curvature | Slower convergence, less numerically stable |
| **Memory inefficiency** | Re-sorts data at every node from scratch | High RAM and CPU thrashing |

Tianqi Chen (then a PhD student at UW) answered all 5 problems simultaneously in XGBoost (2014 paper, open-sourced 2016).

> *"XGBoost is still GBM. But it's GBM the way a Formula 1 car is still a car."*

---

## 2. The Math Upgrade: Newton Boosting (2nd Order Taylor Expansion)

### **A. What GBM Does (1st Order Only)**
Standard GBM adds a tree $h_m$ that fits the **first derivative** (gradient/residual):
$$F_m = F_{m-1} + \eta \cdot h_m, \quad h_m \approx -\frac{\partial L}{\partial F}$$

It only knows the **slope** of the loss function — which direction to move. It doesn't know how steeply curved the landscape is.

### **B. The Taylor Expansion Trick**
XGBoost approximates the loss at step $t$ using the second-order Taylor expansion around the current prediction $\hat{y}^{(t-1)}$:

$$L^{(t)} \approx \sum_{i=1}^{n} \left[ L(y_i, \hat{y}_i^{(t-1)}) + g_i f_t(x_i) + \frac{1}{2} h_i f_t^2(x_i) \right] + \Omega(f_t)$$

Where:
- $g_i = \frac{\partial L(y_i, \hat{y}^{(t-1)})}{\partial \hat{y}^{(t-1)}}$ — the **gradient** (first derivative) for sample $i$.
- $h_i = \frac{\partial^2 L(y_i, \hat{y}^{(t-1)})}{\partial (\hat{y}^{(t-1)})^2}$ — the **Hessian** (second derivative) for sample $i$.
- $f_t(x_i)$ — the prediction of the new tree $t$ for sample $i$.
- $\Omega(f_t)$ — the **regularization term** (penalizes tree complexity).

Since $L(y_i, \hat{y}^{(t-1)})$ is a constant w.r.t. the new tree, we drop it and minimize:
$$\tilde{L}^{(t)} = \sum_{i=1}^{n} \left[ g_i f_t(x_i) + \frac{1}{2} h_i f_t^2(x_i) \right] + \Omega(f_t)$$

### **C. Computing $g_i$ and $h_i$ for Common Losses**

| Loss Function | $g_i$ (Gradient) | $h_i$ (Hessian) |
| :--- | :--- | :--- |
| MSE: $\frac{1}{2}(y-\hat{y})^2$ | $\hat{y}_i - y_i$ | $1$ (constant) |
| Log-Loss (binary): $-y\log p - (1-y)\log(1-p)$ | $p_i - y_i$ | $p_i(1 - p_i)$ |
| Softmax (multiclass) | $p_{ik} - \mathbb{1}[y_i=k]$ | $p_{ik}(1-p_{ik})$ |

**For MSE**: $h_i = 1$ for all $i$ → XGBoost reduces to standard GBM. The Hessian adds power for non-MSE losses.

**For Log-Loss**: $h_i = p_i(1-p_i)$ — this is highest when $p_i = 0.5$ (maximum uncertainty) and lowest near 0 or 1 (high confidence). XGBoost naturally focuses more computation on uncertain samples.

### **D. The Gradient vs. Hessian Analogy**
```
Gradient only (GBM):   "The hill goes DOWN that way. Walk that way."

Gradient + Hessian:    "The hill goes DOWN that way.
                        And it's VERY STEEP — so take a tiny step."
                        OR
                        "The hill goes down that way.
                        And it's flat — so take a big step."
```

The Hessian tells you the **curvature** — how fast the gradient is changing. High curvature → small step (steep cliff). Low curvature → large step (flat gradient). This is Newton's Method applied to function optimization.

---

## 3. The Regularization Upgrade: Built into the Objective

### **A. Standard GBM Regularization**
GBM regularizes **after** building the tree — via `max_depth`, `min_samples_leaf`, `ccp_alpha`. The tree grows greedily, then gets pruned.

### **B. XGBoost Regularization — Inside the Objective**
XGBoost builds regularization **into the loss function being minimized**:

$$\Omega(f_t) = \gamma T + \frac{1}{2}\lambda \sum_{j=1}^{T} w_j^2$$

- $T$ — number of leaves in the tree.
- $w_j$ — the **weight (predicted value)** of leaf $j$.
- $\gamma$ — penalty per leaf (controls number of leaves).
- $\lambda$ — L2 penalty on leaf weights (controls magnitude of predictions).

Additionally, $\alpha$ provides L1 regularization: $+\alpha \sum_j |w_j|$.

### **C. The Structure Score (Similarity Score)**
After substituting the optimal leaf weights into $\tilde{L}^{(t)}$ (by taking derivative and setting to zero), the optimal weight for leaf $j$ is:
$$w_j^* = -\frac{G_j}{H_j + \lambda}$$

Where $G_j = \sum_{i \in \text{leaf } j} g_i$ and $H_j = \sum_{i \in \text{leaf } j} h_i$.

The resulting minimum loss for a given tree structure:
$$\tilde{L}^* = -\frac{1}{2} \sum_{j=1}^T \frac{G_j^2}{H_j + \lambda} + \gamma T$$

The **Structure Score** for each leaf (negated loss):
$$\text{Score}_j = \frac{G_j^2}{H_j + \lambda}$$

### **D. The Gain Formula**
When evaluating whether to split a node with ($G, H$) into left ($G_L, H_L$) and right ($G_R, H_R$):

$$\text{Gain} = \frac{1}{2}\left[\frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{(G_L+G_R)^2}{H_L+H_R+\lambda}\right] - \gamma$$

Breaking this down:
- The first three fraction terms: Score(Left) + Score(Right) − Score(Parent) = how much the split improves the objective.
- $-\gamma$: Only split if improvement exceeds the gamma tax.

**If Gain < 0: Do NOT split.** XGBoost uses this as built-in pruning **during tree growth** — not after.

### **E. Effect of $\lambda$ on Leaf Weights**
Look at $w^* = -G/(H + \lambda)$:
- $\lambda = 0$: $w^* = -G/H$ — no smoothing.
- $\lambda > 0$: The denominator grows → $|w^*|$ shrinks → leaf predictions are **pulled toward 0**.
- This is L2 regularization applied directly to the leaf output values, preventing extreme predictions in any single leaf.

---

## 4. Split Finding: Exact Greedy vs. Histogram Approximation

### **A. The Problem with Naive Split Finding**
To find the best split for a feature with $n$ unique values: sort $n$ values ($O(n \log n)$), then scan all $n-1$ candidate thresholds. For $F$ features and $N$ rows:
$$\text{Cost per tree level} = O(N \cdot F \cdot \log N)$$
For $N=1M, F=500$: extremely slow.

### **B. Exact Greedy Algorithm (XGBoost default for small data)**
Pre-sort each feature column once per training run and store in **Column Blocks** (compressed sparse column format). Within each block, data is sorted by feature value.

For each node split:
1. Scan through pre-sorted feature values in the block.
2. Accumulate $G$ and $H$ sums left-to-right.
3. Compute Gain at each candidate threshold.
4. Pick the threshold with maximum Gain (above $\gamma$).

**Parallelism:** Multiple features can be scanned simultaneously across CPU cores since different column blocks are independent.

### **C. Weighted Quantile Sketch (Approximate, for large data)**
For massive datasets ($N \geq 10M$), even scanning sorted blocks is expensive. XGBoost uses the **Weighted Quantile Sketch** to propose ~33 candidate thresholds per feature:

1. Compute the Hessian $h_i$ for each sample (from the current model state).
2. Define a weighted rank function where each sample has weight $h_i$:
   $$r_k(z) = \frac{\sum_{i: x_{ik} < z} h_i}{\sum_i h_i}$$
3. Find values $z$ such that $r_k(z_{p+1}) - r_k(z_p) < \epsilon$ (uniform in Hessian-weighted rank space).
4. Propose $\approx 1/\epsilon$ candidate splits.

**Why weight by Hessian?**
- High $h_i$ → high curvature → model is uncertain about this sample → **more splits proposed in this region**.
- Low $h_i$ → flat loss → model is confident → fewer splits needed.
- The algorithm concentrates its search effort where the loss landscape is most complex.

**Practical impact:** `tree_method='hist'` (histogram method) groups $n$ unique values into `max_bin` (default 256) bins. Only 256 thresholds evaluated instead of $n$. 10x–100x faster for large datasets. This is also how LightGBM works (LightGBM refined this further).

---

## 5. Sparsity-Aware Split Finding: Native Missing Value Handling

### **The Problem**
Standard GBM fails on `NaN` values — you must impute before training. But imputation loses information: the fact that a value is missing is often predictive itself.

### **XGBoost's Solution: Learned Default Direction**
At every internal node during training, XGBoost runs **two extra scans**:
1. Send all samples with missing values to the **Left child**. Compute the Gain.
2. Send all samples with missing values to the **Right child**. Compute the Gain.
3. **Store the direction that gave higher Gain** as the default direction for this node.

During prediction: if a sample's feature is `NaN`, it follows the learned default direction stored in that node.

### **Why This Is Better Than Imputation**
- **No information loss:** Missingness pattern is learned, not discarded.
- **Context-sensitive:** Different nodes can have different default directions. In one subtree, missing `income` might go right. In another, left — because the meaning of missingness depends on the other features in that branch.
- **Works with sparse matrices:** All zero entries in a sparse matrix are treated as "missing" → XGBoost handles NLP bag-of-words features efficiently without dense storage.

### **min_child_weight Parameter**
This is the **minimum sum of Hessians** ($\sum h_i$) required in a child node to allow a split. Since $h_i$ acts as a weight per sample:
- Low Hessian samples (high confidence) contribute little weight → they need many such samples to meet `min_child_weight`.
- High Hessian samples (uncertain) contribute more weight → fewer uncertain samples needed.
- Effect: blocks splits that don't have enough "uncertain" samples in the child — a principled alternative to `min_samples_leaf`.

---

## 6. System Design: The Engineering Upgrades

### **A. Column Block Structure**
XGBoost pre-sorts each feature column **once** at the start of training and stores it in a Column Block (Compressed Sparse Column format):
- Each block holds: sorted feature values + their corresponding gradient/Hessian statistics.
- **Split finding becomes a linear scan** through pre-sorted blocks instead of re-sorting each time.
- Multiple column blocks can be scanned in **parallel across CPU threads** — even though trees are sequential, the split evaluation within each tree is parallelized.

```
Standard GBM per node:  Sort feature column → scan → find split = O(n log n + n)
XGBoost per node:       Scan pre-sorted block → find split = O(n)   [pre-sort done once]
```

### **B. Cache-Aware Access**
Modern CPUs are 100x faster when reading data from L1/L2 cache vs. RAM. Standard GBM accesses data in a non-sequential pattern (index lookup = cache miss).

XGBoost assigns a **thread buffer** per CPU thread and packs gradient/Hessian pairs sequentially for cache-line-aligned access. Each thread processes its own contiguous chunk → near-zero cache misses.

### **C. Out-of-Core Computation**
For datasets larger than RAM: XGBoost divides data into compressed blocks on disk, loads them in chunks, and processes with block-threaded reads using prefetching. Allows training on datasets that don't fit in memory — something Scikit-Learn GBM cannot do.

### **D. GPU Support**
`tree_method='gpu_hist'` moves the histogram computation to the GPU. All bins' gradient and Hessian sums are computed in a single GPU pass. For large datasets with many features: 10x–50x faster than CPU.

---

## 7. Key Hyperparameters: The Full Panel

XGBoost has many parameters organized into three groups:

### **A. Boosting Parameters**
| Parameter | Default | Role | tuning direction |
| :--- | :---: | :--- | :--- |
| `n_estimators` | 100 | Number of trees | Tune with early stopping |
| `learning_rate` (`eta`) | 0.3 | Step size shrinkage | ↓ lr → need ↑ trees |
| `objective` | `reg:squarederror` | Loss function | Set to task |

### **B. Tree Parameters**
| Parameter | Default | Role | Effect |
| :--- | :---: | :--- | :--- |
| `max_depth` | 6 | Max tree depth | ↑ → ↓ bias, ↑ variance |
| `min_child_weight` | 1 | Min Hessian sum in child | ↑ → more conservative |
| `gamma` ($\gamma$) | 0 | Min gain to split | ↑ → simpler trees |
| `subsample` | 1.0 | Row subsampling fraction | 0.6–0.9 for regularization |
| `colsample_bytree` | 1.0 | Feature fraction per tree | 0.5–0.9, like RF |
| `colsample_bylevel` | 1.0 | Feature fraction per level | Additional randomness |
| `max_leaves` | 0 (unlimited) | Max leaves (leaf-wise growth) | Use for LightGBM-style |

### **C. Regularization Parameters**
| Parameter | Default | Role | Effect |
| :--- | :---: | :--- | :--- |
| `lambda` ($\lambda$) | 1 | L2 penalty on leaf weights | Shrinks leaf values toward 0 |
| `alpha` ($\alpha$) | 0 | L1 penalty on leaf weights | Sparsity — zeros out weak leaves |
| `tree_method` | `auto` | Split algorithm | `'hist'` for large data, `'gpu_hist'` for GPU |

### **D. Practical Tuning Order**
1. Fix `learning_rate=0.1`, tune `n_estimators` with early stopping.
2. Tune `max_depth` ∈ [3, 10] and `min_child_weight` ∈ [1, 10] together.
3. Tune `subsample` and `colsample_bytree` ∈ [0.5, 1.0].
4. Tune `gamma`, `lambda`, `alpha` for regularization.
5. Finally: reduce `learning_rate`, increase `n_estimators` proportionally.

---

## 8. Pros, Cons & When to Use

### **Pros**
1. **Best accuracy on tabular data (2014–2018):** Won majority of Kaggle tabular competitions.
2. **Built-in regularization:** $\gamma$, $\lambda$, $\alpha$ baked into objective — much harder to accidentally overfit than GBM.
3. **Native missing value handling:** No preprocessing required for `NaN`.
4. **Extremely flexible:** Works for regression, binary, multi-class, ranking, survival analysis, custom objectives.
5. **Early stopping built-in:** `early_stopping_rounds` parameter with `eval_set`.
6. **SHAP values natively supported:** `model.get_booster().predict(dmatrix, pred_contribs=True)` returns SHAP values directly.

### **Cons**
1. **Memory-intensive:** Pre-sorted column blocks require RAM proportional to $N \times F$. Can be limiting for very wide datasets.
2. **Many hyperparameters:** At least 8–10 important parameters to tune, with complex interactions.
3. **Slower than LightGBM on large datasets:** LightGBM's leaf-wise growth and refined histogram method give ~3x speedup on large data.
4. **Sequential trees:** Cannot parallelize tree construction (same as GBM). GPU helps within a tree, but not across trees.
5. **Not ideal for very small datasets:** Boosting can overfit on < 1000 samples; consider Random Forest or Logistic Regression.

### **Use When**
✅ Structured tabular data, any scale.
✅ Data has missing values (handles natively).
✅ Need state-of-the-art accuracy with proper tuning.
✅ Sparse data (NLP features, one-hot encoded).

### **Don't Use When**
❌ Extremely large data > 50M rows (prefer LightGBM).
❌ Very fast inference needed (hundreds of deep trees are slow).
❌ Images, text (neural networks).
❌ Tiny datasets (< 500 rows) — will overfit.

---

## 9. GBM vs. XGBoost vs. Random Forest: The Master Comparison

| Feature | Random Forest | GBM (Scikit-Learn) | XGBoost |
| :--- | :---: | :---: | :---: |
| **Training order** | Parallel | Sequential | Sequential |
| **Primary error reduced** | Variance | Bias | Bias + regularized |
| **Overfitting risk** | Low | Medium | Low (with tuning) |
| **Missing values** | ❌ Must impute | ❌ Must impute | ✅ Native |
| **Built-in regularization** | None | None | L1 + L2 + $\gamma$ |
| **Split algorithm** | Exact | Exact | Exact or Histogram |
| **Speed on large data** | Fast (parallel trees) | Slow | Fast (blocks) |
| **Feature importance** | MDI | MDI | MDI + Gain + Cover + SHAP |
| **Extrapolation** | ❌ | ❌ | ❌ |
| **Hyperparameters to tune** | ~4 | ~6 | ~10 |
| **GPU support** | ❌ | ❌ | ✅ |

---

## 10. Interview Deep Dive (12 Questions)

### **Q1: What are the three major upgrades of XGBoost over vanilla GBM?**
**A:** (1) **Math:** Second-order Taylor expansion — uses both gradient ($g_i$) and Hessian ($h_i$). More precise step direction with curvature information. (2) **Regularization:** $\Omega(f) = \gamma T + \frac{1}{2}\lambda\sum w_j^2$ built into the objective — prevents overfitting at every split decision. (3) **Systems:** Pre-sorted Column Blocks for parallel split scanning, cache-aware memory access, GPU support, out-of-core computation — 10–100x faster than Scikit-Learn GBM.

### **Q2: Derive the optimal leaf weight formula $w_j^* = -G_j/(H_j + \lambda)$.**
**A:** For a fixed tree structure, the objective for leaf $j$ is: $\tilde{L}_j = G_j w_j + \frac{1}{2}(H_j + \lambda)w_j^2$. This is a quadratic in $w_j$. Take derivative and set to zero: $\frac{d\tilde{L}_j}{dw_j} = G_j + (H_j + \lambda)w_j = 0 \Rightarrow w_j^* = -G_j/(H_j + \lambda)$. The $\lambda$ term in the denominator is the L2 regularization — it shrinks the leaf weight toward zero.

### **Q3: Explain the Structure Score (Similarity Score) and what it measures.**
**A:** Substituting $w_j^*$ back into the objective for leaf $j$: Score$_j = G_j^2/(H_j + \lambda)$. This measures how "good" a leaf is — how well the gradient signals in the leaf are aligned (high $|G_j|$) while controlling prediction magnitude ($\lambda$). The Gain formula computes Score(Left) + Score(Right) − Score(Parent) − $\gamma$. If positive, the split improves the objective more than the gamma regularization tax.

### **Q4: What does $\gamma$ (gamma) control and how does it differ from `max_depth`?**
**A:** `max_depth` is a hard structural limit — no node deeper than this value, regardless of information content. $\gamma$ is a **minimum gain threshold** — a split only happens if Gain $> \gamma$. Even at depth 2, if a split doesn't improve the objective by at least $\gamma$, it's rejected. $\gamma$ is a "principled" pruner based on actual objective improvement. `max_depth` is blunt. Together they're complementary: `max_depth` prevents wild growth, $\gamma$ prevents useless splits within the allowed depth.

### **Q5: How does XGBoost handle missing values? Why is it better than mean imputation?**
**A:** At each node, XGBoost tries sending all missing values to the left child (computes Gain), then to the right (computes Gain), and stores the better direction as the default. During prediction, any NaN follows that learned direction. Better than mean imputation because: (1) missingness itself can be predictive — XGBoost learns this automatically; (2) the optimal direction can differ at every node — context-sensitive; (3) no artificial value is injected that could distort the feature distribution.

### **Q6: What is `min_child_weight` and how does it relate to Hessians?**
**A:** `min_child_weight` is the minimum **sum of Hessians** ($\sum_i h_i$) required in a child node to allow a split. Since $h_i = p_i(1-p_i)$ for log-loss, high-Hessian samples are those near the decision boundary (uncertain). A leaf needs enough uncertain samples to justify further splitting. Increasing `min_child_weight` prevents splits on very small or very low-Hessian groups, acting as a conservative regularizer. It's the principled equivalent of `min_samples_leaf`, but weighted by model uncertainty.

### **Q7: Explain the Weighted Quantile Sketch. Why weight by Hessian?**
**A:** Instead of evaluating all $n$ unique values as split candidates, XGBoost proposes $\approx 1/\epsilon$ candidate thresholds using the weighted quantile sketch. Each sample has weight = its Hessian $h_i$. Quantiles are computed such that each bucket contains equal sum of Hessians (not equal count of samples). High-Hessian regions (uncertain samples) get more candidate splits. Low-Hessian regions (confident samples) get fewer. This concentrates split-finding computation where the loss landscape is most complex — efficient and principled.

### **Q8: Why does `tree_method='hist'` speed up XGBoost dramatically?**
**A:** Instead of scanning all unique feature values, `hist` discretizes each feature into `max_bin` bins (default 256). Gradient and Hessian statistics are accumulated per bin. Split search only evaluates 256 thresholds instead of potentially millions of unique values. Memory usage drops from $O(N)$ to $O(bins)$ per feature. For $N=10M$: ~39,000x fewer threshold evaluations. LightGBM extended this further as its primary algorithm.

### **Q9: XGBoost has four types of feature importance. What are they?**
**A:** (1) **weight**: Number of times a feature is used across all trees — similar to MDI count. (2) **gain**: Average Gain contributed by splits using this feature — measures actual objective improvement. (3) **cover**: Average number of samples affected by splits using this feature — measures reach. (4) **SHAP values**: Model-agnostic exact attribution (most theoretically sound). Use `importance_type='gain'` or SHAP for reliable importance. `weight` and `cover` can be misleading.

### **Q10: How does XGBoost handle multi-class classification?**
**A:** XGBoost builds $K$ trees per boosting round (one per class), using the softmax objective (`objective='multi:softmax'` or `multi:softprob'`). For each class $k$: gradient $g_{ik} = p_{ik} - \mathbb{1}[y_i=k]$ and Hessian $h_{ik} = p_{ik}(1-p_{ik})$. The final prediction converts class scores to probabilities via softmax: $p_{ik} = e^{F_{ik}} / \sum_c e^{F_{ic}}$. Training is $K$× more expensive than binary classification.

### **Q11: When would you choose XGBoost over LightGBM?**
**A:** XGBoost is preferred when: (1) Dataset is medium-sized (< 1M rows) — both are fast, XGBoost may generalize slightly better with exact splits. (2) You need GPU training with the `gpu_hist` method. (3) You need the exact Weighted Quantile Sketch (more theoretically justified). LightGBM wins when: dataset is very large (> 10M rows), you need the fastest training, leaf-wise (depth-first) growth is beneficial, or you have high-cardinality categorical features (LightGBM's native categorical encoding).

### **Q12: Explain the Column Block structure and why it enables parallelism.**
**A:** XGBoost pre-sorts each feature column into a Column Block (CSC format: sorted feature values + corresponding gradient/Hessian at each position) at the start of training. This sorting is done ONCE per training run. During split finding at each node: different CPU threads simultaneously scan different column blocks. Since each column block is independent (different features), there's no data dependency between threads. The critical insight: **the inter-tree sequential constraint (tree M needs tree M-1) does not apply to intra-tree split evaluation** — all feature candidates within one tree level can be scanned in parallel.

---

## 11. Chapter Summary & Interview Checklist

| # | Question | Minimum Expected Answer |
| :- | :--- | :--- |
| 1 | Three upgrades of XGBoost over GBM? | 2nd order math + regularization in objective + system speed. |
| 2 | What is the Hessian and what does it tell you? | 2nd derivative = curvature. High curvature → small step. |
| 3 | Derive optimal leaf weight. | $w^* = -G/(H+\lambda)$. Set derivative of quadratic to zero. |
| 4 | What does $\gamma$ do? How different from `max_depth`? | Gain threshold per split. Max_depth = hard limit. $\gamma$ = info-based pruner. |
| 5 | How does XGBoost handle missing values? | Learns default direction (left/right) per node during training. |
| 6 | What is `min_child_weight`? | Min sum-of-Hessians in child. Principled version of min_samples_leaf. |
| 7 | What is the Weighted Quantile Sketch? | Proposes split candidates weighted by Hessian. More candidates where model is uncertain. |
| 8 | Why is `tree_method='hist'` faster? | Bins features into 256 buckets. Scans 256 thresholds vs. millions of unique values. |
| 9 | Four feature importance types in XGBoost? | weight, gain, cover, SHAP. |
| 10 | XGBoost vs LightGBM choice? | Small-medium: XGBoost. Very large / categoricals: LightGBM. |
| 11 | How does the Column Block enable parallelism? | Pre-sorted once. Different features scanned simultaneously per CPU thread. |
| 12 | What does $\lambda$ (lambda) regularize? | L2 penalty on leaf weights. Shrinks predictions toward zero. |

---

## 12. Implementation Playground (Placeholder)

Five cells — each targeting a specific concept from this chapter.


In [ ]:
# ─── CELL 1: XGBoost vs GBM vs RF — Accuracy & Speed Benchmark ────────────────
# pip install xgboost
import xgboost as xgb
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import time

# TODO: Create a dataset with 50,000 samples and 20 features
pass

# TODO: Time and train all three: XGBClassifier, GradientBoostingClassifier, RandomForestClassifier
# Use same n_estimators (100) and similar depth for fair comparison
pass

# TODO: Print: training time and test accuracy for each
# Expected: XGBoost much faster than Sklearn GBM; accuracy comparable or better
pass

In [ ]:
# ─── CELL 2: Structure Score — Verify the Math Manually ────────────────────────
import numpy as np

# Suppose a binary classification node has these gradient/Hessian sums:
# Parent: G = -3.5, H = 8.0
# Left split:  G_L = -5.0, H_L = 5.0
# Right split: G_R = +1.5, H_R = 3.0
# lambda = 1.0,  gamma = 0.5

G_parent, H_parent = -3.5, 8.0
G_left,   H_left   = -5.0, 5.0
G_right,  H_right  =  1.5, 3.0
lam, gamma = 1.0, 0.5

# TODO: Compute Score_parent, Score_left, Score_right
# Formula: Score = G^2 / (H + lambda)
pass

# TODO: Compute Gain = 0.5 * (Score_left + Score_right - Score_parent) - gamma
pass

# TODO: Print: should we split? (Gain > 0)
# TODO: Compute optimal leaf weights for left and right
# Formula: w* = -G / (H + lambda)
pass

In [ ]:
# ─── CELL 3: Native Missing Value Handling — XGBoost vs Scikit-Learn ───────────
import numpy as np
import xgboost as xgb
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# TODO: Create dataset, then randomly set 20% of values to NaN
pass

# XGBoost: train directly on data with NaN — no imputation needed
# TODO: Train XGBClassifier on X_train (NaN values present)
# TODO: Predict on X_test (NaN values present)
pass

# Sklearn GBM: must impute first
# TODO: Use SimpleImputer(strategy='mean') then train GradientBoostingClassifier
pass

# TODO: Compare accuracy — XGBoost should match or beat imputed GBM
# Key point: XGBoost LEARNS the best direction for NaN, not a fixed fill value
pass

In [ ]:
# ─── CELL 4: Early Stopping + Regularization (gamma, lambda, alpha) ───────────
import xgboost as xgb
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# TODO: Load Breast Cancer dataset, split train/val/test (60/20/20)
pass

# TODO: Train XGBClassifier with early_stopping_rounds=20, eval_set=[(X_val, y_val)]
# Parameters: n_estimators=1000, learning_rate=0.05, max_depth=4
# Print: model.best_iteration (optimal n_estimators found)
pass

# TODO: Experiment with regularization — train 3 models:
#   Model A: gamma=0, lambda=1, alpha=0  (default)
#   Model B: gamma=2, lambda=5, alpha=0  (heavy regularization)
#   Model C: gamma=0, lambda=1, alpha=2  (L1 sparsity)
# Compare test accuracy and number of leaves in each tree (use model.get_booster().trees_to_dataframe())
pass

In [ ]:
# ─── CELL 5: Feature Importance — All Four Types + SHAP ───────────────────────
# pip install shap xgboost
import xgboost as xgb
import shap
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# TODO: Load diabetes regression dataset, split train/test
pass

# TODO: Train XGBRegressor (objective='reg:squarederror')
pass

# TODO: Plot all four importance types side by side:
#   importance_type in ['weight', 'gain', 'cover']
#   Use xgb.plot_importance(model, importance_type=...)
# Observe: Rankings often differ — 'gain' is most informative
pass

# TODO: Compute and plot SHAP summary (beeswarm)
# explainer = shap.TreeExplainer(model)
# shap_values = explainer.shap_values(X_test)
# shap.summary_plot(shap_values, X_test)
# Compare SHAP ranking vs 'gain' ranking
pass